![banner](./images/banner.png)

### Suivi des feux dans les tourbières du bassin du Congo avec FIRMS

Ce notebook suit les **feux actifs détectés par satellite (NASA FIRMS)** dans les tourbières du bassin du Congo, en **République du Congo (COG)** ou en **RD Congo (COD)**. Il croise chaque détection avec les zones administratives, les aires protégées, les concessions forestières et la carte des tourbières de Crezee et al. (2022), puis produit des graphiques, des tableaux et des cartes, ainsi que des fichiers à partager ou à valider sur le terrain.

**Avant de commencer :** saisissez votre clé FIRMS (`MAP_KEY`), puis réglez le pays, la période et les sources dans la cellule de paramètres (étape 0). Exécutez ensuite le notebook de haut en bas. Toutes les étapes après l'étape 0 n'utilisent que les données du pays choisi.

| Section | Ce qu'elle fait | Ce qu'elle produit |
|---|---|---|
| **0. Pays et jeux de données** | Choix du pays, de la période et des sources FIRMS ; chargement des couches SIG et filtrage sur le pays | Tableau de la sélection ; tableau des fichiers SIG utilisés avec leurs attributs d'identification ; tableau des capteurs FIRMS ; carte des classes de tourbière du pays (avec zones administratives, aires protégées et concessions) ; diagramme circulaire de l'occupation du sol |
| **1. Données de feux** | Téléchargement des détections SNPP, NOAA-20, NOAA-21 (VIIRS, 375 m) et MODIS (1 km) par blocs de 5 jours ; suppression des détections de faible confiance ; fusion des capteurs | Nombre de détections par capteur et par bloc ; table combinée des feux |
| **2. Attributs** | Pour chaque feu : zone administrative, aire protégée, concession forestière, classe de tourbière (les feux sans classe sont retirés) et **distance à la tourbe**, calculée à partir du raster des tourbières (classes 4 et 5). **Seuls les feux sur la tourbe ou à moins de 5 km (`ANALYSIS_MAX_DIST_KM`) sont gardés pour la suite** | Carte des points chauds de feux ; graphique du nombre de feux par classe de tourbière ; tableau du nombre de feux par distance à la tourbe et par classe |
| **3. Analyses** | Analyse des feux dans le temps, par niveau de confiance, par rapport aux concessions et aires protégées, et selon la distance à la tourbe | Détections quotidiennes par satellite ; détections quotidiennes par niveau de confiance (un graphique par satellite) ; feux sur la tourbe à l'intérieur / à l'extérieur des **concessions** et des **aires protégées**, **par zone administrative** (graphiques + tableaux avec totaux et %) ; feux par classe de distance à la tourbe et courbe cumulée (% des feux à ≤ 1, 5 et 10 km) ; feux par distance et par zone administrative (graphique + tableau avec distance médiane et % à moins de 1 et 5 km) ; **taille des feux** : détections voisines (≤ 1 km, ≤ 1 jour d'écart) regroupées en feux, avec nombre de feux par superficie et par durée, tableau des plus grands feux touchant la tourbe et export GeoJSON des emprises ; carte des points chauds de feux superposée aux classes de tourbière, avec les emprises des feux touchant la tourbe |
| **4. Export Google Drive** | Export des feux de la zone d'analyse (sur la tourbe ou à moins de 5 km), avec leurs attributs, en CSV dans SEPAL, puis en feuille Google dans le dossier `Congo_Peatlands` (remplacée si elle existe déjà) | Fichier CSV ; feuille Google avec lien cliquable |
| **5. Export pour Ground** | Sélection des feux des `GROUND_LAST_DAYS` derniers jours de la période, situés sur la tourbe (classes 4 et 5) ou à moins de 1 km de la tourbe (`GROUND_MAX_DIST_KM`), chacun nommé d'après sa date d'acquisition | Fichier GeoJSON `fires_peatland_<PAYS>_<début>_<fin>.geojson` pour la validation sur le terrain dans Open Foris Ground |

*Classes de tourbière (Crezee et al. 2022) : 1 eau, 2 savane, 3 forêt de terre ferme, 4 marécage dominé par les palmiers, 5 marécage dominé par les feuillus. Les classes 4 et 5 sont considérées comme de la tourbe.*

In [ ]:
# en cas d'erreur avec les imports, exécuter une fois puis supprimer
#!pip install gspread --break-system-packages

In [ ]:
# en cas d'erreur avec les imports, exécuter une fois puis supprimer
#!pip install gspread google-auth google-auth-oauthlib google-auth-httplib2 --break-system-packages

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from IPython.display import display, HTML
import geopandas as gpd
from matplotlib import pyplot as plt
from matplotlib import font_manager
import geopandas as gpd
from shapely.geometry import box
import rasterio
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
from folium import plugins
from datetime import datetime
from datetime import datetime, date
from datetime import datetime, timedelta
import time
import gspread
import warnings
from io import StringIO
import json
from tqdm import tqdm
import shapely.geometry
from rasterio.sample import sample_gen
from pathlib import Path
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.credentials import Credentials

# --- ajoutés pour le choix du pays, la carte des tourbières et la distance à la tourbe ---
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform
from rasterio.windows import bounds as window_bounds
from affine import Affine
from shapely.geometry.base import BaseGeometry
from rasterio.crs import CRS
from scipy import ndimage
from scipy.spatial import cKDTree

vous devez vous inscrire et obtenir votre propre clé sur https://firms.modaps.eosdis.nasa.gov/api/map_key/ n'utilisez pas la mienne !

In [ ]:
# saisir la clé MAP_KEY reçue par e-mail - ne pas supprimer les guillemets
MAP_KEY = "1ccacc221fd0778472a2ea1e55f3e4ae"

In [ ]:
# interroger data_availability pour connaître la période disponible pour chaque jeu de données
# cette URL renvoie les informations sur tous les capteurs pris en charge et leurs jeux de données
# au lieu de 'all', vous pouvez indiquer un capteur, ex. : LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
firms_data = pd.read_csv(da_url)
display(firms_data)

#### 0. Choisir le pays et les jeux de données

**Cellule suivante : paramètres à modifier.** Indiquez `COUNTRY` = **"COG"** (République du Congo) ou **"COD"** (RD Congo) et la période (`start_date`, `end_date`), puis exécutez le notebook de haut en bas. Toutes les étapes suivantes n'utilisent que les données de ce pays.

Les sources FIRMS (archive `_SP` ou quasi temps réel `_NRT`) se règlent dans la cellule d'après.

In [ ]:
# =====================================================================
# PARAMÈTRES À MODIFIER : pays et période
# =====================================================================

# Pays à analyser : "COG" = République du Congo, "COD" = République démocratique du Congo
COUNTRY = "COD"

# Période des observations de feux (dates incluses) aaaa-mm-jj
start_date = "2026-01-01"
end_date   = "2026-05-01"

# Export GeoJSON pour Ground (section 5) : nombre de jours exportés, en remontant depuis end_date
# (end_date incluse). None = toute la période.
GROUND_LAST_DAYS = 7

In [ ]:
# Sources FIRMS : *_SP = standard/archive (périodes passées), *_NRT = quasi temps réel (~2 derniers mois)
FIRMS_SOURCES = {
    "SNPP":   "VIIRS_SNPP_SP",     # ou "VIIRS_SNPP_NRT"
    "MODIS":  "MODIS_SP",          # ou "MODIS_NRT"
    "NOAA20": "VIIRS_NOAA20_SP",   # ou "VIIRS_NOAA20_NRT"
    "NOAA21": "VIIRS_NOAA21_NRT",  # NRT uniquement (depuis 2024)
}

# ---------------------------------------------------------------------
COUNTRY_NAMES = {"COG": "République du Congo", "COD": "République démocratique du Congo"}
COUNTRY = COUNTRY.strip().upper()
if COUNTRY not in COUNTRY_NAMES:
    raise ValueError(f"COUNTRY doit être 'COG' ou 'COD', et non '{COUNTRY}'")
if datetime.strptime(start_date, "%Y-%m-%d") > datetime.strptime(end_date, "%Y-%m-%d"):
    raise ValueError("start_date doit être antérieure à end_date")
# style de tableau compact utilisé pour les résumés
def compact_table(df, title=None):
    css = ("border-collapse:collapse;font-size:12px;margin:2px 0 10px 0;")
    html = df.to_html(index=False, border=0)
    html = html.replace('<table class="dataframe">', f'<table style="{css}">')
    html = html.replace("<th>", '<th style="text-align:left;padding:3px 10px;background:#eef2f5;border-bottom:1px solid #ccc;">')
    html = html.replace("<td>", '<td style="text-align:left;padding:3px 10px;border-bottom:1px solid #eee;">')
    head = f'<div style="font-weight:bold;font-size:13px;margin-top:6px;">{title}</div>' if title else ""
    return head + html

display(HTML(compact_table(pd.DataFrame([{
    "Sélection": COUNTRY,
    "Pays": COUNTRY_NAMES[COUNTRY],
    "Du": start_date,
    "Au": end_date,
}]))))

# hauteur des cartes (pixels) : augmenter si la carte est trop petite
MAP_HEIGHT = 750

def show_map(m, height=None):
    """Affiche une carte folium dans un cadre de hauteur fixe (sans défilement)."""
    fig = folium.Figure(width="100%", height=f"{height or MAP_HEIGHT}px")
    fig.add_child(m)
    return fig

In [ ]:
# zones et périmètres à évaluer
GIS_DIR = "/home/sepal-user/Peatlands_Congo/gis_data"

adm1_path = f"{GIS_DIR}/peatlands_adm1_aoi.shp"
pas_path = f"{GIS_DIR}/peatlands_PAs.shp"
conc_cod_path = f"{GIS_DIR}/forest_concessions_COD.shp"
conc_cog_path = f"{GIS_DIR}/forest_concessions_COG.shp"

# classes d'occupation du sol des tourbières (Crezee et al. 2022)
peat_types = f"{GIS_DIR}/Crezee_2022_class_100m_4bit.tif"

# distance à la tourbe : calculée dans ce notebook à partir du raster d'occupation du sol des tourbières
PEAT_CODES = [4, 5]      # classes considérées comme tourbe : marécages dominés par les palmiers et par les feuillus
DIST_RES_M = 100         # résolution (m) de la grille utilisée pour le calcul des distances
DIST_MARGIN_KM = 50      # recherche de tourbe jusqu'à cette distance au-delà de la frontière (tourbe du pays voisin)
ANALYSIS_MAX_DIST_KM = 5  # zone d'analyse : feux sur la tourbe ou à moins de cette distance (km) — appliqué après le calcul des distances

In [ ]:
# Classes de tourbière : communes à la carte, aux graphiques et aux tables attributaires
PEAT_CLASS_NAMES = {
    1: "eau",
    2: "savane",
    3: "forêt de terre ferme",
    4: "marécage dominé par les palmiers",
    5: "marécage dominé par les feuillus",
}
PEAT_CLASS_COLORS = {
    1: "#1f78b4",   # eau
    2: "#f5e663",   # savane
    3: "#2e8b57",   # forêt de terre ferme
    4: "#c084fc",   # marécage dominé par les palmiers
    5: "#6b21a8",   # marécage dominé par les feuillus
}

# couche des concessions et champs (attributs) du nom et du type, par pays
# un nom de champ, une liste (le premier trouvé est utilisé) ou None = détection automatique ;
# les champs utilisés sont affichés dans le tableau des fichiers SIG
CONC_DATASETS = {
    "COD": {"path": conc_cod_path, "name": ["attributai", "NOM"], "type": "desc_type"},
    "COG": {"path": conc_cog_path, "name": ["attributai", "NOM"], "type": "desc_type"},
}

# champs du nom et du type des aires protégées (None = détection automatique)
PA_FIELDS = {"name": "NAME", "type": None}

# noms de champs recherchés lors de la détection automatique (sans tenir compte de la casse)
NAME_CANDIDATES = ["attributai", "NAME", "NOM", "ORIG_NAME", "nom_ap", "nom_conc", "concession", "nom_ufa", "ufa",
                   "titulaire", "societe", "company"]
TYPE_CANDIDATES = ["DESIG", "DESIG_ENG", "DESIG_TYPE", "desc_type", "type", "type_conc", "categorie",
                   "category", "statut", "status", "IUCN_CAT"]

In [ ]:
# Zone d'intérêt (AOI) des tourbières
aoi = gpd.read_file(f"{GIS_DIR}/Peatlands_AOI.shp")

In [ ]:
# ---------------------------------------------------------------------
# Charger les couches et ne garder que le pays choisi
# ---------------------------------------------------------------------
def find_col(gdf, candidates, required=True):
    """Renvoie la première colonne de gdf correspondant à l'un des candidats (sans tenir compte de la casse)."""
    lower = {c.lower(): c for c in gdf.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise KeyError(f"Aucune des colonnes {candidates} trouvée. Colonnes disponibles : {list(gdf.columns)}")
    return None

def pick_field(gdf, configured, candidates):
    """Champ indiqué dans les paramètres, sinon le premier champ candidat trouvé (ou None)."""
    if configured:
        wanted = configured if isinstance(configured, (list, tuple)) else [configured]
        col = find_col(gdf, wanted, required=False)
        if col:
            return col
        print(f"⚠️ Champ(s) {wanted} introuvable(s) ; détection automatique utilisée à la place.")
    return find_col(gdf, candidates, required=False)

def union_geom(gdf):
    try:
        return gdf.union_all()      # GeoPandas >= 0.14
    except AttributeError:
        return gdf.unary_union

# --- Zones administratives ---
ADM1_all = gpd.read_file(adm1_path).to_crs(4326)
ISO_COL = find_col(ADM1_all, ["iso3_code", "iso3", "ISO3", "adm0_iso", "GID_0"])
ADM1_NAME_COL = find_col(ADM1_all, ["gaul1_name", "adm1_name", "ADM1_NAME", "NAME_1", "name"])

ADM1 = ADM1_all[ADM1_all[ISO_COL] == COUNTRY].copy()
if ADM1.empty:
    raise ValueError(f"Aucune zone administrative avec {ISO_COL} == '{COUNTRY}' dans {adm1_path}")
COUNTRY_GEOM = union_geom(ADM1)

# --- AOI des tourbières découpée au pays (contour sur les cartes uniquement) ---
aoi = aoi.to_crs(4326)
try:
    AOI = gpd.clip(aoi, ADM1)
except Exception:
    AOI = aoi

# --- Aires protégées du pays ---
PAs_all = gpd.read_file(pas_path).to_crs(4326)
PAs = PAs_all[PAs_all.intersects(COUNTRY_GEOM)].copy()

# --- Concessions forestières du pays ---
CONC_PATH = CONC_DATASETS[COUNTRY]["path"]
CONC = gpd.read_file(CONC_PATH).to_crs(4326)
CONC = CONC[CONC.intersects(COUNTRY_GEOM)].copy()
CONC_NAME_COL = pick_field(CONC, CONC_DATASETS[COUNTRY]["name"], NAME_CANDIDATES)
CONC_TYPE_COL = pick_field(CONC, CONC_DATASETS[COUNTRY]["type"], TYPE_CANDIDATES)
CONC_ATTR = CONC_NAME_COL or CONC_TYPE_COL

# --- Emprise (bbox) des requêtes FIRMS (minx, miny, maxx, maxy) ---
minx, miny, maxx, maxy = ADM1.total_bounds
bbox = f"{minx:.4f},{miny:.4f},{maxx:.4f},{maxy:.4f}"

# ---------------------------------------------------------------------
# Résumé : fichiers SIG utilisés pour la suite de l'analyse et
# attributs qui identifient leurs entités
# ---------------------------------------------------------------------
PA_NAME_COL = pick_field(PAs, PA_FIELDS["name"], NAME_CANDIDATES)
PA_TYPE_COL = pick_field(PAs, PA_FIELDS["type"], TYPE_CANDIDATES)

for _layer, _gdf, _what, _col in [
    ("Aires protégées", PAs, "nom", PA_NAME_COL), ("Aires protégées", PAs, "type", PA_TYPE_COL),
    ("Concessions forestières", CONC, "nom", CONC_NAME_COL), ("Concessions forestières", CONC, "type", CONC_TYPE_COL),
]:
    if _col is None:
        print(f"⚠️ {_layer} : aucun champ « {_what} » trouvé. Champs disponibles : {[c for c in _gdf.columns if c != 'geometry']}. Indiquez-le dans les paramètres.")

def name_type_examples(gdf, name_col, type_col):
    parts = [examples(gdf, name_col, 4)]
    if type_col:
        parts.append("types : " + examples(gdf, type_col, 4))
    return " | ".join(p for p in parts if p)

def examples(gdf, col, n=6):
    if col is None or gdf.empty:
        return ""
    vals = sorted(gdf[col].dropna().astype(str).unique())
    more = f" … (+{len(vals) - n} autres)" if len(vals) > n else ""
    return ", ".join(vals[:n]) + more

def short(p):
    """gis_data/<nom du fichier> au lieu du chemin complet"""
    return f"{Path(GIS_DIR).name}/{Path(p).name}"

with rasterio.open(peat_types) as _src:
    peat_desc = f"{_src.width} x {_src.height} px, {_src.crs}"

gis_files = pd.DataFrame([
    ("Zones administratives (ADM1)", short(adm1_path),
     f"{len(ADM1)} sur {len(ADM1_all)}",
     f"{ISO_COL} = '{COUNTRY}' (filtre pays) ; {ADM1_NAME_COL} (nom de la zone administrative)",
     examples(ADM1, ADM1_NAME_COL)),
    ("Aires protégées", short(pas_path),
     f"{len(PAs)} sur {len(PAs_all)}",
     f"nom : {PA_NAME_COL or '—'} ; type : {PA_TYPE_COL or '—'}",
     name_type_examples(PAs, PA_NAME_COL, PA_TYPE_COL)),
    (f"Concessions forestières ({COUNTRY})", short(CONC_PATH),
     f"{len(CONC)}",
     f"nom : {CONC_NAME_COL or '—'} ; type : {CONC_TYPE_COL or '—'}",
     name_type_examples(CONC, CONC_NAME_COL, CONC_TYPE_COL)),
    ("AOI des tourbières (contour de carte uniquement)", short(f"{GIS_DIR}/Peatlands_AOI.shp"),
     f"{len(AOI)}", "", ""),
    ("Occupation du sol des tourbières (raster)", short(peat_types),
     peat_desc,
     "valeur du pixel = code de la classe",
     ", ".join(f"{k} {v}" for k, v in PEAT_CLASS_NAMES.items())),
    ("Distance à la tourbe (calculée)", short(peat_types),
     f"grille de {DIST_RES_M} m, pays + {DIST_MARGIN_KM} km",
     "distance au pixel le plus proche des classes de tourbe " + " / ".join(str(c) for c in PEAT_CODES),
     ", ".join(PEAT_CLASS_NAMES[c] for c in PEAT_CODES)),
], columns=["Couche", "Fichier", "Entités dans le pays", "Attribut(s) d'identification", "Valeurs (exemples)"])

firms = pd.DataFrame(
    [(k, v) for k, v in FIRMS_SOURCES.items()], columns=["Capteur", "Source FIRMS"]
)

display(HTML(
    compact_table(gis_files, f"Fichiers SIG inclus : {COUNTRY_NAMES[COUNTRY]} ({COUNTRY})")
    + compact_table(firms, "Détections de feux : capteurs disponibles")
    + f'<div style="font-size:11px;color:grey;">Zone de recherche FIRMS (bbox) : {bbox}</div>'
))

##### Occupation du sol des tourbières dans le pays choisi

In [ ]:
# (PEAT_CLASS_NAMES et PEAT_CLASS_COLORS sont définis dans la cellule des chemins ci-dessus)

MAX_PX = 1500   # taille max. de l'image affichée (le raster est sous-échantillonné pour l'affichage uniquement)
SIMPLIFY_TOL = 0.002   # simplification des contours affichés (~200 m) pour une carte plus légère

from contextlib import nullcontext
with rasterio.open(peat_types) as src:
    nodata = src.nodata
    in_wgs84 = src.crs is not None and src.crs.to_epsg() == 4326      # déjà en WGS 84 : lecture directe, sans reprojection
    with (nullcontext(src) if in_wgs84 else WarpedVRT(src, crs="EPSG:4326", resampling=Resampling.nearest)) as vrt:
        # emprise du pays, limitée à l'emprise du raster
        cminx, cminy, cmaxx, cmaxy = ADM1.total_bounds
        L, B, R, T = vrt.bounds
        cminx, cminy, cmaxx, cmaxy = max(cminx, L), max(cminy, B), min(cmaxx, R), min(cmaxy, T)

        win = from_bounds(cminx, cminy, cmaxx, cmaxy, transform=vrt.transform)
        scale = max(win.width, win.height) / MAX_PX
        scale = max(scale, 1)
        out_h, out_w = int(win.height / scale), int(win.width / scale)

        peat_arr = vrt.read(1, window=win, out_shape=(out_h, out_w), resampling=Resampling.nearest)
        out_transform = window_transform(win, vrt.transform) * Affine.scale(win.width / out_w, win.height / out_h)
        left, bottom, right, top = window_bounds(win, vrt.transform)

# découpage selon la frontière du pays
inside = geometry_mask([COUNTRY_GEOM], out_shape=peat_arr.shape, transform=out_transform, invert=True)
peat_arr = np.where(inside, peat_arr, 0)
if nodata is not None:
    peat_arr[peat_arr == nodata] = 0

# coloration des classes (transparent là où il n'y a pas de classe)
rgba = np.zeros((*peat_arr.shape, 4), dtype=np.uint8)
for code_, hexcol in PEAT_CLASS_COLORS.items():
    r, g, b = [int(255 * v) for v in mcolors.to_rgb(hexcol)]
    rgba[peat_arr == code_] = [r, g, b, 220]

# --- diagramme circulaire : part de chaque classe dans le pays (à partir de l'image affichée, approximatif) ---
vals, counts = np.unique(peat_arr[peat_arr > 0], return_counts=True)
keep = [i for i, v in enumerate(vals) if int(v) in PEAT_CLASS_NAMES]
vals, counts = vals[keep], counts[keep]
pct = 100 * counts / counts.sum()
labels = [PEAT_CLASS_NAMES[int(v)] for v in vals]

fig, ax = plt.subplots(figsize=(8, 5))
wedges, _, autotexts = ax.pie(
    counts,
    colors=[PEAT_CLASS_COLORS[int(v)] for v in vals],
    autopct=lambda p: f"{p:.1f}%" if p >= 3 else "",
    pctdistance=0.75, startangle=90, counterclock=False,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
for t, v in zip(autotexts, vals):
    t.set_color("white" if int(v) in (1, 3, 5) else "black")
    t.set_fontsize(10)
ax.legend(wedges, [f"{l} ({p:.1f}%)" for l, p in zip(labels, pct)],
          loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
ax.set_title(f"Occupation du sol des tourbières : {COUNTRY_NAMES[COUNTRY]} ({COUNTRY})")
ax.axis("equal")
plt.tight_layout()
plt.show()

# conserver l'image et son emprise pour la carte des points chauds plus loin
PEAT_RGBA = rgba
PEAT_BOUNDS = [[bottom, left], [top, right]]

# --- carte ---
m_peat = folium.Map(
    location=[(bottom + top) / 2, (left + right) / 2],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
)
folium.raster_layers.ImageOverlay(
    image=rgba,
    bounds=[[bottom, left], [top, right]],
    mercator_project=True,
    name=f"Classes de tourbière ({COUNTRY})",
).add_to(m_peat)

folium.GeoJson(
    ADM1[[ADM1_NAME_COL, "geometry"]].assign(geometry=ADM1.geometry.simplify(SIMPLIFY_TOL)), name="Zones administratives",
    style_function=lambda x: {"color": "white", "weight": 1.5, "fill": False},
    tooltip=folium.GeoJsonTooltip(fields=[ADM1_NAME_COL], aliases=["Zone administrative"]),
).add_to(m_peat)

def outline_layer(gdf, name_col, type_col, layer_name, colour, weight):
    cols = [c for c in [name_col, type_col] if c]
    aliases = [a for c, a in [(name_col, "Nom"), (type_col, "Type")] if c]
    folium.GeoJson(
        gdf[cols + ["geometry"]].assign(geometry=gdf.geometry.simplify(SIMPLIFY_TOL)), name=layer_name, show=False,
        style_function=lambda x: {"color": colour, "weight": weight, "fill": False},
        tooltip=folium.GeoJsonTooltip(fields=cols, aliases=aliases) if cols else None,
    ).add_to(m_peat)

if not PAs.empty:
    outline_layer(PAs, PA_NAME_COL, PA_TYPE_COL, "Aires protégées", "orange", 1.5)
if not CONC.empty:
    outline_layer(CONC, CONC_NAME_COL, CONC_TYPE_COL, "Concessions forestières", "#7CFC00", 1)

legend_items = "".join(
    f'<div><span style="display:inline-block;width:12px;height:12px;background:{PEAT_CLASS_COLORS[k]};'
    f'margin-right:6px;"></span>{v}</div>'
    for k, v in PEAT_CLASS_NAMES.items()
)
m_peat.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:20px;left:10px;z-index:9999;background:white;'
    f'padding:8px 10px;border:1px solid grey;font-size:12px;">'
    f'<b>Classes de tourbière : {COUNTRY}</b>{legend_items}</div>'
))
folium.LayerControl().add_to(m_peat)
plugins.Fullscreen().add_to(m_peat)
show_map(m_peat)

#### 1. obtenir les données de feux

In [ ]:
# Colonnes FIRMS attendues (utilisées quand une source ne renvoie rien)
viirs_cols = [
    "latitude","longitude","bright_ti4","scan","track",
    "acq_date","acq_time","satellite","instrument",
    "confidence","version","bright_ti5","frp","daynight"
]
modis_cols = [
    "latitude","longitude","brightness","scan","track",
    "acq_date","acq_time","satellite","instrument",
    "confidence","version","bright_t31","frp","daynight"
]

def fetch_firms(source, label, bbox, start_date, end_date, columns, chunk=5, min_conf=30, pause=2):
    """Télécharge les détections FIRMS pour bbox de start_date à end_date (incluses).

    Les requêtes sont faites par blocs de 5 jours maximum (limite de l'API). Dans l'API FIRMS « area »,
    la date est le DÉBUT du bloc : elle renvoie [DATE .. DATE + DAY_RANGE - 1].
    Les détections de faible confiance sont supprimées (VIIRS 'l', MODIS < min_conf).
    """
    cur = datetime.strptime(start_date, "%Y-%m-%d")
    stop = datetime.strptime(end_date, "%Y-%m-%d")
    dfs = []

    while cur <= stop:
        days = min(chunk, (stop - cur).days + 1)
        date_str = cur.strftime("%Y-%m-%d")
        url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{MAP_KEY}/{source}/{bbox}/{days}/{date_str}"
        )
        print(f"🔹 {label} : {days} jours à partir du {date_str}")

        try:
            r = requests.get(url, timeout=60)
            text = r.text.strip()

            if r.status_code != 200:
                print(f"⚠️ HTTP {r.status_code}, ignoré.")
            elif not text or "latitude" not in text.splitlines()[0]:
                print(f"⚠️ Pas de données / message de l'API : {text[:150]}")
            else:
                df = pd.read_csv(StringIO(text))
                n_all = len(df)
                if "confidence" in df.columns and not df.empty:
                    if not pd.api.types.is_numeric_dtype(df["confidence"]):   # VIIRS : l / n / h
                        df = df[~df["confidence"].astype(str).str.lower().isin(["l", "low"])]
                    else:
                        df = df[df["confidence"] >= min_conf]
                if not df.empty:
                    print(f"✅ {len(df)} enregistrements récupérés")
                    dfs.append(df)
                elif n_all:
                    print("⚠️ Uniquement des détections de faible confiance, ignorées.")
                else:
                    print("⚠️ Pas de données pour cet intervalle.")

        except Exception as e:
            print(f"❌ Échec de la requête : {e}")

        cur += timedelta(days=days)
        time.sleep(pause)

    if dfs:
        out = pd.concat(dfs, ignore_index=True).drop_duplicates()
    else:
        out = pd.DataFrame(columns=columns)
    out["satellite"] = label
    print(f"\n🔥 {label} : {len(out)} enregistrements au total")
    return out

In [ ]:
all_SNPP = fetch_firms(FIRMS_SOURCES["SNPP"], "SNPP", bbox, start_date, end_date, viirs_cols)

In [ ]:
all_SNPP.head()

In [ ]:
all_MODIS = fetch_firms(FIRMS_SOURCES["MODIS"], "MODIS", bbox, start_date, end_date, modis_cols)

In [ ]:
all_MODIS.head()

In [ ]:
all_NOAA20 = fetch_firms(FIRMS_SOURCES["NOAA20"], "NOAA20", bbox, start_date, end_date, viirs_cols)

In [ ]:
all_NOAA21 = fetch_firms(FIRMS_SOURCES["NOAA21"], "NOAA21", bbox, start_date, end_date, viirs_cols)

##### combiner toutes les détections de feux

In [ ]:
# Combiner
merged_fires = pd.concat([all_MODIS, all_NOAA21, all_NOAA20, all_SNPP], ignore_index=True)

merged_fires_gdf = gpd.GeoDataFrame(
    merged_fires,
    geometry=gpd.points_from_xy(merged_fires.longitude, merged_fires.latitude),
    crs="EPSG:4326"
)
len(merged_fires_gdf)

In [ ]:
# supprimer les doublons
# S'assurer que acq_date est bien au format date
merged_fires_gdf["acq_date"] = pd.to_datetime(merged_fires_gdf["acq_date"])

# Trier par date décroissante (la plus récente en premier)
merged_fires_gdf = merged_fires_gdf.sort_values("acq_date", ascending=False)

# Supprimer les doublons par coordonnées en gardant la plus récente
fires_unique = merged_fires_gdf.drop_duplicates(subset=["latitude", "longitude"], keep="first")

print(f"🔥 Détections de feux uniques : {len(fires_unique)}")

#### 2. extraire et ajouter les attributs

In [ ]:
# Couches déjà chargées et filtrées sur le pays choisi à l'étape 0
layers = {
    "ADM1": ADM1,
    "PAs": PAs,
    f"CONC_{COUNTRY}": CONC,
}

for name, gdf in layers.items():
    print(f"\nChamps attributaires de {name} ({len(gdf)} entités)")
    print("-" * (len(name) + 36))
    for col in [c for c in gdf.columns if c != "geometry"]:
        print(f"• {col}")

In [ ]:
# Ne garder que les feux situés dans les zones administratives du pays choisi
merged_fires_gdf = merged_fires_gdf.to_crs(ADM1.crs)

inside_idx = gpd.sjoin(
    merged_fires_gdf[["geometry"]], ADM1[["geometry"]], how="inner", predicate="within"
).index.unique()
fires_within_adm1 = merged_fires_gdf.loc[inside_idx].copy()

print(f"🔥 Feux dans les zones administratives de {COUNTRY} : {len(fires_within_adm1):,}")

In [ ]:
# --- Passer en WGS84 pour Folium ---
fires_within_adm1 = fires_within_adm1.to_crs(epsg=4326)

# S'assurer que la colonne existe
if "confidence_num" not in fires_within_adm1.columns:
    fires_within_adm1["confidence_num"] = 1.0
else:
    fires_within_adm1["confidence_num"] = fires_within_adm1["confidence_num"].fillna(1.0)

# --- Gérer un jeu de données vide ---
if fires_within_adm1.empty:
    center_lat, center_lon = 0, 0
else:
    center_lat = fires_within_adm1.geometry.y.mean()
    center_lon = fires_within_adm1.geometry.x.mean()

# --- Fond de carte sombre ---
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors",
    max_zoom=16,
)

# --- 🔥 Préparer les poids des points chauds de feux ---
if "confidence" in fires_within_adm1.columns:

    fires_within_adm1["confidence_num"] = pd.to_numeric(
        fires_within_adm1["confidence"], errors="coerce"
    )

    # Si les valeurs de confiance sont catégorielles (type VIIRS)
    if fires_within_adm1["confidence_num"].isna().all():

        conf_map = {"low": 0.5, "nominal": 1.0, "medium": 1.0, "high": 2.0}

        fires_within_adm1["confidence_num"] = (
            fires_within_adm1["confidence"]
            .astype(str)
            .str.lower()
            .map(conf_map)
            .fillna(1.0)
        )

    else:
        fires_within_adm1["confidence_num"] = fires_within_adm1["confidence_num"].fillna(1.0)

else:
    fires_within_adm1["confidence_num"] = 1.0

# --- Points chauds : feux regroupés sur une grille d'environ 1 km (pondérés par la confiance) ---
_g = fires_within_adm1.geometry
_heat = pd.DataFrame({"lat": (_g.y / 0.01).round() * 0.01, "lon": (_g.x / 0.01).round() * 0.01,
                      "w": fires_within_adm1["confidence_num"].astype(float).values})
_heat = _heat.groupby(["lat", "lon"], as_index=False).w.sum()
_heat["w"] = (_heat.w / max(_heat.w.quantile(0.95), 1e-9)).clip(upper=1)
heat_data = _heat[["lat", "lon", "w"]].round(4).values.tolist()

# --- Ajouter les points chauds de feux seulement s'il y a des données ---
if heat_data:
    plugins.HeatMap(
        heat_data,
        radius=8,
        blur=15,
        max_zoom=9,
        min_opacity=0.3,
    ).add_to(m)

# --- Outils de carte optionnels ---
plugins.Fullscreen().add_to(m)
# minicarte (encart) avec les tuiles Esri : les tuiles OpenStreetMap par défaut sont bloquées dans les notebooks
plugins.MiniMap(
    tile_layer=folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
    ),
    toggle_display=True,
).add_to(m)
plugins.LocateControl(auto_start=False).add_to(m)

show_map(m)

##### ajouter les classes de tourbière à partir du raster dans SEPAL

In [ ]:
with rasterio.open(peat_types) as src:
    # Aligner le SCR sur celui du raster
    fires_within_adm1 = fires_within_adm1.to_crs(src.crs)

    coords = [(pt.x, pt.y) for pt in fires_within_adm1.geometry]

    batch_size = 50000
    peat_vals = []

    nodata = src.nodata  # peut être None

    for i in tqdm(range(0, len(coords), batch_size), desc="Échantillonnage"):
        batch_coords = coords[i:i + batch_size]

        for v in src.sample(batch_coords):
            val = v[0]  # première bande

            # Convertir nodata en NaN
            if nodata is not None and val == nodata:
                peat_vals.append(np.nan)
            else:
                peat_vals.append(val)

assert len(peat_vals) == len(fires_within_adm1), "Nombre de valeurs échantillonnées incorrect."

# Ajouter les résultats (tous les feux sont conservés ; peat_class_code peut être NaN)
fires_within_adm1["peat_class_code"] = peat_vals
fires_within_adm1["has_peat_class"] = fires_within_adm1["peat_class_code"].notna().astype(int)

In [ ]:
# Convertir les codes numériques en libellés (PEAT_CLASS_NAMES est défini dans la cellule des chemins)
fires_within_adm1["peat_class_name"] = fires_within_adm1["peat_class_code"].map(PEAT_CLASS_NAMES)

# Remplacer les valeurs manquantes ou nulles par "pas de données"
fires_within_adm1["peat_class_name"] = fires_within_adm1["peat_class_name"].fillna("pas de données")
fires_within_adm1.loc[
    fires_within_adm1["peat_class_code"] == 0,
    "peat_class_name"
] = "pas de données"

# --- Supprimer les feux sans classe de tourbière (hors du raster ou nodata) ---
n_before = len(fires_within_adm1)
fires_within_adm1 = fires_within_adm1[fires_within_adm1["peat_class_name"] != "pas de données"].copy()
print(f"{n_before - len(fires_within_adm1):,} feux sans classe de tourbière supprimés ; {len(fires_within_adm1):,} restants")
fires_within_adm1["peat_class_name"].value_counts()

In [ ]:
warnings.filterwarnings("ignore", message="Glyph 128293")
# --- Nombre de feux par classe de tourbière (NaN inclus) ---
peat_counts = (
    fires_within_adm1["peat_class_name"]
    .value_counts(dropna=False)
    .rename_axis("Classe")
    .reset_index(name="Nombre de feux")
)
peat_counts["Classe"] = peat_counts["Classe"].fillna("pas de données")

# --- Mêmes couleurs que la carte des tourbières ---
color_map = {name: PEAT_CLASS_COLORS[code_] for code_, name in PEAT_CLASS_NAMES.items()}
color_map["pas de données"] = "grey"
peat_counts["Couleur"] = peat_counts["Classe"].str.lower().map(color_map).fillna("grey")

# --- Graphique ---
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(
    peat_counts["Classe"],
    peat_counts["Nombre de feux"],
    color=peat_counts["Couleur"]
)
ax.bar_label(bars, labels=[f"{v:,}" for v in peat_counts["Nombre de feux"]],
             label_type="edge", padding=3, fontsize=10, color="black")

ax.set_title(f"Nombre de feux par classe de tourbière : {COUNTRY}", fontsize=14)
ax.set_xlabel("Classe de tourbière")
ax.set_ylabel("Nombre de feux")
plt.xticks(rotation=30, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

##### calculer la distance à la tourbe

Pour chaque feu, distance (km) au pixel de tourbe le plus proche (classes de `PEAT_CODES`, par défaut 4 marécage dominé par les palmiers et 5 marécage dominé par les feuillus) dans le raster d'occupation du sol des tourbières. Les feux situés sur la tourbe reçoivent 0.

Fonctionnement : le raster est rééchantillonné sur une grille de `DIST_RES_M` dans une projection conservant les distances, centrée sur le pays et couvrant le pays plus `DIST_MARGIN_KM`, afin de tenir compte de la tourbe située de l'autre côté de la frontière. Les pixels de bordure des taches de tourbe sont extraits et chaque feu est associé au pixel de bordure le plus proche. Le raster des tourbières étant à 100 m, `DIST_RES_M = 100` utilise sa pleine résolution ; la précision est d'environ ±50 m.

In [ ]:
# Classes de distance (km) : tranches de 500 m jusqu'à 5 km, puis de 5 km, jusqu'à ANALYSIS_MAX_DIST_KM.
# La première classe (0) = feu sur la tourbe
_edges = [e for e in [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 10, 15, 20, 25] if e < ANALYSIS_MAX_DIST_KM] + [ANALYSIS_MAX_DIST_KM]
DIST_BINS = [-0.001, 0] + _edges
_n = lambda v: f"{v:g}".replace(".", ",")
def _band(a, b):
    if b < 1:
        return f"{a * 1000:.0f}-{b * 1000:.0f} m"
    if a < 1:
        return f"{a * 1000:.0f} m-{_n(b)} km"
    return f"{_n(a)}-{_n(b)} km"
DIST_LABELS = ["0 (sur tourbe)"] + [_band(a, b) for a, b in zip([0] + _edges[:-1], _edges)]
assert len(DIST_LABELS) == len(DIST_BINS) - 1

# --- projection conservant les distances, centrée sur le pays (azimutale équidistante, mètres) ---
c = COUNTRY_GEOM.centroid
DIST_CRS = CRS.from_proj4(f"+proj=aeqd +lat_0={c.y:.4f} +lon_0={c.x:.4f} +datum=WGS84 +units=m +no_defs")

# --- grille couvrant le pays + la marge ---
gminx, gminy, gmaxx, gmaxy = ADM1.to_crs(DIST_CRS).total_bounds
m_ = DIST_MARGIN_KM * 1000
gminx, gminy, gmaxx, gmaxy = gminx - m_, gminy - m_, gmaxx + m_, gmaxy + m_
gw = int(np.ceil((gmaxx - gminx) / DIST_RES_M))
gh = int(np.ceil((gmaxy - gminy) / DIST_RES_M))
grid_transform = Affine(DIST_RES_M, 0, gminx, 0, -DIST_RES_M, gmaxy)
print(f"Grille de distance : {gw:,} x {gh:,} px à {DIST_RES_M} m")

with rasterio.open(peat_types) as src:
    with WarpedVRT(src, crs=DIST_CRS, transform=grid_transform, width=gw, height=gh,
                   resampling=Resampling.nearest) as vrt:
        grid = vrt.read(1)

# --- masque de tourbe et pixels de bordure des taches de tourbe ---
peat_mask = np.isin(grid, PEAT_CODES)
del grid
edge = peat_mask & ~ndimage.binary_erosion(peat_mask)
rows, cols = np.nonzero(edge)
del edge
print(f"Pixels de tourbe : {peat_mask.sum():,}  |  pixels de bordure : {len(rows):,}")

fires_within_adm1 = fires_within_adm1.to_crs(4326)

if len(rows) == 0:
    print("⚠️ Pas de tourbe dans la grille : distance mise à NaN")
    dist_m = np.full(len(fires_within_adm1), np.nan)
else:
    # centres des pixels de bordure (m) -> arbre KD -> pixel de bordure le plus proche de chaque feu
    ex = grid_transform.c + (cols + 0.5) * grid_transform.a
    ey = grid_transform.f + (rows + 0.5) * grid_transform.e
    tree = cKDTree(np.column_stack([ex, ey]))

    fp = fires_within_adm1.to_crs(DIST_CRS)
    fxy = np.column_stack([fp.geometry.x, fp.geometry.y])
    dist_m, _ = tree.query(fxy, workers=-1)

    # feux sur la tourbe = 0 : d'après la classe échantillonnée à pleine résolution ci-dessus,
    # ou d'après la grille (feu dans un pixel de tourbe de la grille)
    r = np.floor((fxy[:, 1] - grid_transform.f) / grid_transform.e).astype(int)
    k = np.floor((fxy[:, 0] - grid_transform.c) / grid_transform.a).astype(int)
    in_grid = (r >= 0) & (r < gh) & (k >= 0) & (k < gw)
    on_peat_grid = np.zeros(len(fxy), dtype=bool)
    on_peat_grid[in_grid] = peat_mask[r[in_grid], k[in_grid]]
    on_peat = fires_within_adm1["peat_class_code"].isin(PEAT_CODES).values | on_peat_grid
    dist_m = np.where(on_peat, 0.0, dist_m)

del peat_mask

fires_within_adm1["dist_peat_km"] = np.round(dist_m / 1000, 3)
fires_within_adm1["dist_class"] = (
    pd.cut(fires_within_adm1["dist_peat_km"], bins=DIST_BINS, labels=DIST_LABELS)
    .astype(str).replace("nan", "pas de données")
)

# --- Zone d'analyse : ne garder que les feux sur la tourbe ou à moins de ANALYSIS_MAX_DIST_KM de la tourbe ---
n_all = len(fires_within_adm1)
fires_within_adm1 = fires_within_adm1[fires_within_adm1["dist_peat_km"] <= ANALYSIS_MAX_DIST_KM].copy()
n_on = int((fires_within_adm1["dist_peat_km"] == 0).sum())
print(f"Zone d'analyse : {len(fires_within_adm1):,} feux conservés sur {n_all:,} — {n_on:,} sur la tourbe, "
      f"{len(fires_within_adm1) - n_on:,} à moins de {ANALYSIS_MAX_DIST_KM:g} km ; {n_all - len(fires_within_adm1):,} feux plus loin retirés")

print(fires_within_adm1["dist_peat_km"].describe().round(2))

# Contrôle : classe de distance vs classe de tourbière
display(HTML("<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Nombre de feux par distance à la tourbe</div>"))
display(pd.crosstab(fires_within_adm1["peat_class_name"], fires_within_adm1["dist_class"])
        .reindex(columns=[l for l in DIST_LABELS + ["pas de données"] if l in set(fires_within_adm1["dist_class"])]))

In [ ]:
# ------------------------------------------------------------------
# Joindre les attributs avec un préfixe.
# Un feu situé sur une limite commune ou dans des polygones superposés serait
# dupliqué par la jointure ; seule la première correspondance est gardée
# pour que chaque feu ne soit compté qu'une fois.
# ------------------------------------------------------------------
def spatial_join_all_prefix(base_gdf, join_gdf, prefix, predicate="intersects"):
    join_gdf = join_gdf.copy()
    join_gdf.drop(columns=["Shape_Area", "Shape_Leng"], errors="ignore", inplace=True)

    joined = gpd.sjoin(base_gdf, join_gdf, how="left", predicate=predicate)
    joined = joined[~joined.index.duplicated(keep="first")]

    join_cols = [c for c in join_gdf.columns if c != "geometry"]
    rename_dict = {c: f"{prefix}_{c}" for c in join_cols if c in joined.columns}
    joined.rename(columns=rename_dict, inplace=True)

    joined.drop(columns=["index_right"], errors="ignore", inplace=True)
    return joined

def spatial_join_some_prefix(base_gdf, join_gdf, columns, prefix, predicate="intersects"):
    join_gdf = join_gdf.copy()

    # Correspondance des colonnes demandées sans tenir compte de la casse
    col_map = {c.lower(): c for c in join_gdf.columns}
    matched = [col_map[c.lower()] for c in columns if c.lower() in col_map]

    join_gdf = join_gdf[["geometry"] + matched].copy()

    joined = gpd.sjoin(base_gdf, join_gdf, how="left", predicate=predicate)
    joined = joined[~joined.index.duplicated(keep="first")]

    rename_dict = {c: f"{prefix}_{c}" for c in matched if c in joined.columns}
    joined.rename(columns=rename_dict, inplace=True)

    joined.drop(columns=["index_right"], errors="ignore", inplace=True)
    return joined

fires_within_adm1 = fires_within_adm1.to_crs(ADM1.crs)

# ------------------------------------------------------------------
# Appliquer les jointures (les couches sont déjà limitées au pays choisi)
# ------------------------------------------------------------------
fires_all_atts = spatial_join_all_prefix(fires_within_adm1, ADM1, "ADM")

ADM_COL = f"ADM_{ADM1_NAME_COL}"          # admin area name / nom de la zone administrative

# ------------------------------------------------------------------
# Aires protégées et concessions : nom et type pour chaque feu situé à l'intérieur.
# Si un feu tombe dans plusieurs polygones superposés, les noms sont listés, séparés par « ; ».
# ------------------------------------------------------------------
def join_name_type(base, layer, name_col, type_col, prefix, flag):
    lay = layer[["geometry"]].copy()
    lay[f"{prefix}_name"] = layer[name_col] if name_col else None
    lay[f"{prefix}_type"] = layer[type_col] if type_col else None
    hits = gpd.sjoin(base[["geometry"]], lay, how="inner", predicate="intersects")

    def join_vals(v):
        v = sorted(set(v.dropna().astype(str)))
        return " ; ".join(v) if v else None

    agg = hits.groupby(level=0)[[f"{prefix}_name", f"{prefix}_type"]].agg(join_vals)
    out = base.drop(columns=[f"{prefix}_name", f"{prefix}_type", flag], errors="ignore").join(agg)
    out[flag] = out.index.isin(agg.index).astype(int)
    return out

fires_all_atts = join_name_type(fires_all_atts, PAs, PA_NAME_COL, PA_TYPE_COL, "PA", "PA")
fires_all_atts = join_name_type(fires_all_atts, CONC, CONC_NAME_COL, CONC_TYPE_COL, "conc", "for_conc")

# Champs indicateurs (1 = à l'intérieur, 0 = à l'extérieur) : PA et for_conc
print(fires_all_atts[["PA", "for_conc"]].sum().to_string())

fires_all_atts.head(5)

In [ ]:
print("\n".join(fires_all_atts.columns))

#### 3. Évaluer et visualiser

In [ ]:
fire_counts = fires_all_atts["satellite"].value_counts().reset_index()
fire_counts.columns = ["satellite", "Nombre de feux"]

print("🔥 Détections de feux par source :")
print(fire_counts)

fires_all_atts["acq_date"] = pd.to_datetime(fires_all_atts["acq_date"], errors="coerce")

daily_counts = fires_all_atts.groupby(["acq_date", "satellite"]).size().reset_index(name="Nombre de feux")

warnings.filterwarnings("ignore", message="Glyph .* missing from font")

plt.figure(figsize=(14, 6))
for src in daily_counts["satellite"].unique():
    data = daily_counts[daily_counts["satellite"] == src]
    plt.plot(data["acq_date"], data["Nombre de feux"], marker="o", markersize=3, label=src)

plt.title("Détections de feux quotidiennes par satellite")
plt.xlabel("Date")
plt.ylabel("Nombre de feux")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Type de détection")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------
# Convertir la date d'acquisition
# -------------------------------------------------------
fires_all_atts["acq_date"] = pd.to_datetime(
    fires_all_atts["acq_date"], errors="coerce"
)

# -------------------------------------------------------
# Harmoniser les valeurs de confiance
# -------------------------------------------------------
fires_all_atts["confidence_num"] = pd.to_numeric(
    fires_all_atts["confidence"], errors="coerce"
)

# Confiance numérique (MODIS)
fires_all_atts["confidence_class"] = pd.cut(
    fires_all_atts["confidence_num"],
    bins=[-np.inf, 30, 80, np.inf],
    labels=["Faible", "Nominale", "Élevée"],
    right=False
)

# Remplacer la confiance catégorielle VIIRS
fires_all_atts.loc[
    fires_all_atts["confidence"].astype(str).str.lower().isin(["l", "low"]),
    "confidence_class"
] = "Faible"

fires_all_atts.loc[
    fires_all_atts["confidence"].astype(str).str.lower().isin(["n", "nominal"]),
    "confidence_class"
] = "Nominale"

fires_all_atts.loc[
    fires_all_atts["confidence"].astype(str).str.lower().isin(["h", "high"]),
    "confidence_class"
] = "Élevée"

# -------------------------------------------------------
# Paramètres des graphiques
# -------------------------------------------------------
confidence_order = ["Faible", "Nominale", "Élevée"]

colors = {
    "Faible": "#fdbb84",
    "Nominale": "orange",
    "Élevée": "red"
}

satellites = fires_all_atts["satellite"].dropna().unique()

warnings.filterwarnings("ignore", message="Glyph .* missing from font")

# -------------------------------------------------------
# Graphique par satellite
# -------------------------------------------------------
for sat in satellites:

    sat_df = fires_all_atts[fires_all_atts["satellite"] == sat]

    # Nombre de feux par date + confiance
    daily_conf_counts = (
        sat_df
        .groupby(["acq_date", "confidence_class"], observed=True)
        .size()
        .reset_index(name="Nombre de feux")
    )

    pivot_df = (
        daily_conf_counts
        .pivot(index="acq_date", columns="confidence_class", values="Nombre de feux")
        .fillna(0)
    )

    pivot_df = pivot_df.reindex(
        columns=[c for c in confidence_order if c in pivot_df.columns]
    )

    # ---------------------------------------------------
    # Barres empilées
    # ---------------------------------------------------
    fig, ax = plt.subplots(figsize=(14, 6))

    bottom = pd.Series(0, index=pivot_df.index)

    for conf in pivot_df.columns:
        ax.bar(
            pivot_df.index,
            pivot_df[conf],
            bottom=bottom,
            color=colors.get(conf, "gray"),
            label=conf
        )
        bottom += pivot_df[conf]

    # ---------------------------------------------------
    # Mise en forme des axes
    # ---------------------------------------------------
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%Y"))

    plt.xticks(rotation=45, ha="right")

    plt.title(f"Détections de feux quotidiennes par niveau de confiance — {sat}")
    plt.xlabel("Date")
    plt.ylabel("Nombre de feux")

    plt.legend(
        title="Confiance",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Feux sur la tourbe (PEAT_CODES : 4 marécage à palmiers, 5 marécage à feuillus) ---
fires_peat = fires_all_atts[fires_all_atts["peat_class_code"].isin(PEAT_CODES)].copy()
print(f"Feux sur la tourbe : {len(fires_peat):,} sur {len(fires_all_atts):,} en {COUNTRY}")

OUTSIDE = "Feux sur tourbe à l’extérieur"
UNKNOWN_TYPE = "Intérieur (type non renseigné)"
PCT_IN = "% intérieur"

def by_adm_and_type(df, flag_col, type_col):
    """Admin area x (one column per type + outside), with total and % inside."""
    cat = pd.Series(
        np.where(df[flag_col] == 1, df[type_col].fillna(UNKNOWN_TYPE), OUTSIDE),
        index=df.index,
    )
    t = pd.crosstab(df[ADM_COL].fillna("non attribué"), cat)
    types = sorted(c for c in t.columns if c != OUTSIDE)
    t = t.reindex(columns=types + [OUTSIDE], fill_value=0)
    t["Total"] = t[types + [OUTSIDE]].sum(axis=1)
    t[PCT_IN] = (100 * (t["Total"] - t[OUTSIDE]) / t["Total"]).round(1)
    t.index.name = "Zone administrative"
    t.columns.name = None
    return t.sort_values("Total"), types

def with_total(t):
    t = t.sort_values("Total", ascending=False)
    num = [c for c in t.columns if c != PCT_IN]
    tot = t[num].sum()
    tot[PCT_IN] = round(100 * (tot["Total"] - tot[OUTSIDE]) / tot["Total"], 1) if tot["Total"] else 0.0
    out = pd.concat([t, tot.to_frame(f"TOTAL {COUNTRY}").T])
    return out.astype({c: int for c in num})

if fires_peat.empty:
    print("⚠️ Aucun feu dans les classes de tourbe pour ce pays et cette période.")
else:
    for label, flag in [("aires protégées", "PA"), ("concessions forestières", "for_conc")]:
        n_in = int(fires_peat[flag].sum())
        print(f"  {label} : {n_in:,} à l'intérieur, {len(fires_peat) - n_in:,} à l'extérieur")

    conc_tab, conc_types = by_adm_and_type(fires_peat, "for_conc", "conc_type")
    pa_tab, pa_types = by_adm_and_type(fires_peat, "PA", "PA_type")
    # contrôle : les tableaux ne comptent que les feux sur la tourbe
    assert conc_tab["Total"].sum() == pa_tab["Total"].sum() == len(fires_peat)

    n_adm = len(conc_tab)
    fig, axes = plt.subplots(1, 2, figsize=(18, max(5, 0.5 * n_adm + 3.5)))

    for ax, tab, types, title, cmap_name, legend_title in [
        (axes[0], conc_tab, conc_types, "des concessions forestières", "Greens", "Type de concession"),
        (axes[1], pa_tab, pa_types, "des aires protégées", "Oranges", "Type d'aire protégée"),
    ]:
        cmap = plt.get_cmap(cmap_name)
        colours = [cmap(0.45 + 0.5 * i / max(len(types) - 1, 1)) for i in range(len(types))] + ["lightgrey"]
        tab[types + [OUTSIDE]].plot(kind="barh", stacked=True, ax=ax, color=colours, width=0.75)
        ax.bar_label(
            ax.containers[-1],
            labels=[f"{t:,} ({p:.0f} % à l’intérieur)" for t, p in zip(tab["Total"], tab[PCT_IN])],
            padding=3, fontsize=9,
        )
        ax.set_title(f"Feux sur la tourbe à l’intérieur / à l’extérieur {title}\npar zone administrative et par type : {COUNTRY}")
        ax.set_xlabel("Nombre de feux")
        ax.set_ylabel("")
        ax.margins(x=0.3)
        ax.grid(axis="x", linestyle="--", alpha=0.4)
        ax.legend(title=legend_title, loc="upper center", bbox_to_anchor=(0.5, -0.27),
                  fontsize=11, title_fontsize=12, frameon=False,
                  handlelength=2.5, handleheight=1.5, labelspacing=0.6)

    plt.tight_layout()
    plt.show()

    display(HTML("<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Concessions forestières : feux sur la tourbe par zone administrative et par type de concession</div>"))
    display(with_total(conc_tab))
    display(HTML("<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Aires protégées : feux sur la tourbe par zone administrative et par type d'aire protégée</div>"))
    display(with_total(pa_tab))

##### Feux par distance à la tourbe

In [ ]:
dist_order = DIST_LABELS + ["pas de données"]
dfd = fires_all_atts.copy()
dfd["dist_class"] = pd.Categorical(dfd["dist_class"], categories=dist_order, ordered=True)

# couleurs : rouge foncé = sur ou près de la tourbe, clair = loin
cmap = plt.get_cmap("YlOrRd_r")
dist_colors = [cmap(i / (len(DIST_LABELS))) for i in range(len(DIST_LABELS))] + ["lightgrey"]

# --- 1. Tous les feux du pays par classe de distance + courbe cumulée ---
fig, axes = plt.subplots(1, 2, figsize=(19, 5.5), gridspec_kw={"width_ratios": [3, 2]})

counts = dfd["dist_class"].value_counts().reindex(dist_order, fill_value=0)
bars = axes[0].bar(counts.index.astype(str), counts.values, color=dist_colors)
axes[0].bar_label(bars, labels=[f"{v:,}" for v in counts.values], padding=3, fontsize=8)
axes[0].set_title(f"Feux par distance à la tourbe : {COUNTRY}")
axes[0].set_ylabel("Nombre de feux")
axes[0].tick_params(axis="x", rotation=45)
plt.setp(axes[0].get_xticklabels(), ha="right")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

d = np.sort(dfd["dist_peat_km"].dropna().values)
if len(d):
    axes[1].plot(d, 100 * np.arange(1, len(d) + 1) / len(d), color="firebrick")
    axes[1].set_xlim(0, min(50, d.max()) if d.max() > 0 else 1)
    summary_lines = []
    for km in [1, 5, 10]:
        pct = 100 * (d <= km).mean()
        axes[1].axvline(km, color="grey", linestyle=":", lw=1)
        axes[1].plot(km, pct, "o", color="firebrick", ms=4)
        summary_lines.append(f"≤ {km:>2} km : {pct:4.0f} % des feux")
    axes[1].text(0.97, 0.05, "\n".join(summary_lines), transform=axes[1].transAxes,
                 ha="right", va="bottom", fontsize=9, family="monospace",
                 bbox=dict(boxstyle="round", facecolor="white", edgecolor="lightgrey"))
axes[1].set_title("Part cumulée des feux selon la distance à la tourbe")
axes[1].set_xlabel("Distance à la tourbe (km)")
axes[1].set_ylabel("% des feux")
axes[1].grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# --- 2. Par zone administrative ---
dist_by_adm = pd.crosstab(dfd[ADM_COL].fillna("non attribué"), dfd["dist_class"], dropna=False)
dist_by_adm = dist_by_adm.reindex(columns=dist_order, fill_value=0)
dist_by_adm = dist_by_adm.loc[dist_by_adm.sum(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(13, max(4, 0.5 * len(dist_by_adm) + 2)))
dist_by_adm.plot(kind="barh", stacked=True, ax=ax, color=dist_colors, width=0.75)
ax.set_title(f"Feux par distance à la tourbe et par zone administrative : {COUNTRY}")
ax.set_xlabel("Nombre de feux")
ax.set_ylabel("")
ax.legend(title="Distance à la tourbe", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

# --- 3. Tableau : effectifs par classe + distance médiane et part à moins de 1 / 5 km par zone administrative ---
stats = dfd.groupby(dfd[ADM_COL].fillna("non attribué"))["dist_peat_km"].agg(
    mediane_km="median",
    pct_moins_1km=lambda s: 100 * (s <= 1).mean(),
    pct_moins_5km=lambda s: 100 * (s <= 5).mean(),
).round(1)
dist_table = dist_by_adm.join(stats)
dist_table.insert(len(dist_order), "Total", dist_by_adm.sum(axis=1))
display(HTML("<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Nombre de feux par distance à la tourbe et par zone administrative</div>"))
display(dist_table.sort_values("Total", ascending=False))

##### Taille des feux : regroupement des détections

Une même zone qui brûle est souvent détectée par plusieurs pixels voisins, par plusieurs satellites et pendant plusieurs jours. Ici, les détections sont regroupées en **feux** : deux détections appartiennent au même feu si elles sont à moins de `FIRE_LINK_DIST_M` (1 km par défaut, soit des pixels voisins) **et** à au plus `FIRE_MAX_GAP_DAYS` jour d'écart (1 par défaut). Les liens se font de proche en proche, si bien qu'un feu qui progresse jour après jour reste un seul feu.

Pour chaque feu : dates de début et de fin, durée, nombre de détections, capteurs, puissance radiative (FRP totale et maximale, en MW), part des détections sur la tourbe, distance minimale à la tourbe, zone administrative, aire protégée et concession, et **superficie**, calculée comme l'union des pixels détectés (taille de pixel `scan` × `track`).

**Limites :** la superficie des pixels n'est pas une surface brûlée. Un petit feu suffit à déclencher un pixel entier (375 m pour VIIRS, 1 km pour MODIS), ce qui peut surestimer la superficie. À l'inverse, les nuages, la fumée et les feux de tourbe couvants peu chauds peuvent échapper à la détection. À utiliser comme indicateur de taille relative, et à vérifier avec des images (Sentinel-2, Planet) pour les feux importants.

In [ ]:
# Paramètres du regroupement des détections en feux
FIRE_LINK_DIST_M = 1000    # deux détections sont voisines si elles sont à moins de cette distance (m)
FIRE_MAX_GAP_DAYS = 1      # ... et à au plus ce nombre de jours d'écart
SIZE_BINS = [0, 1, 5, 25, 100, np.inf]                                  # km²
SIZE_LABELS = ["< 1 km²", "1-5 km²", "5-25 km²", "25-100 km²", "> 100 km²"]
DUR_BINS = [0, 1, 2, 5, 10, np.inf]                                     # jours
DUR_LABELS = ["1 jour", "2 jours", "3-5 jours", "6-10 jours", "> 10 jours"]

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

ev = fires_all_atts.copy()
ev["acq_date"] = pd.to_datetime(ev["acq_date"], errors="coerce")
ev = ev[ev["acq_date"].notna()].copy()
ev["frp"] = pd.to_numeric(ev.get("frp"), errors="coerce")
ev["on_peat"] = ev["peat_class_code"].isin(PEAT_CODES)

if ev.empty:
    print("⚠️ Aucune détection à regrouper.")
else:
    # --- relier les détections voisines dans l'espace et dans le temps, puis former les groupes (feux) ---
    evp = ev.to_crs(DIST_CRS)
    xy = np.column_stack([evp.geometry.x, evp.geometry.y])
    day = (ev["acq_date"] - ev["acq_date"].min()).dt.days.values
    pairs = cKDTree(xy).query_pairs(FIRE_LINK_DIST_M, output_type="ndarray")
    pairs = pairs[np.abs(day[pairs[:, 0]] - day[pairs[:, 1]]) <= FIRE_MAX_GAP_DAYS]
    n = len(ev)
    graph = coo_matrix((np.ones(len(pairs)), (pairs[:, 0], pairs[:, 1])), shape=(n, n))
    n_events, labels = connected_components(graph, directed=False)
    ev["fire_event_id"] = labels

    # --- emprise de chaque feu : union des pixels détectés (scan x track, en km) ---
    default_km = np.where(ev["satellite"].eq("MODIS"), 1.0, 0.375)
    half_x = pd.to_numeric(ev["scan"], errors="coerce").fillna(pd.Series(default_km, index=ev.index)).values * 500
    half_y = pd.to_numeric(ev["track"], errors="coerce").fillna(pd.Series(default_km, index=ev.index)).values * 500
    footprints = gpd.GeoDataFrame(
        {"fire_event_id": labels},
        geometry=[box(x - hx, y - hy, x + hx, y + hy) for (x, y), hx, hy in zip(xy, half_x, half_y)],
        crs=DIST_CRS,
    ).dissolve("fire_event_id")

    # --- attributs de chaque feu ---
    def join_unique(s):
        v = sorted(set(str(x) for x in s.dropna() for x in str(x).split(" ; ")))
        return " ; ".join(v) if v else None

    g = ev.groupby("fire_event_id")
    events = pd.DataFrame({
        "start": g["acq_date"].min(),
        "end": g["acq_date"].max(),
        "n_detections": g.size(),
        "n_days": g["acq_date"].nunique(),
        "sensors": g["satellite"].agg(lambda s: ", ".join(sorted(s.dropna().astype(str).unique()))),
        "frp_total_MW": g["frp"].sum(min_count=1).round(1),
        "frp_max_MW": g["frp"].max().round(1),
        "pct_on_peat": (100 * g["on_peat"].mean()).round(1),
        "min_dist_peat_km": g["dist_peat_km"].min(),
        "adm": g[ADM_COL].agg(lambda s: s.mode().iat[0] if s.notna().any() else None),
        "PA_name": g["PA_name"].agg(join_unique),
        "conc_name": g["conc_name"].agg(join_unique),
    })
    events["duration_days"] = (events["end"] - events["start"]).dt.days + 1
    events["area_km2"] = (footprints.area / 1e6).reindex(events.index).round(2)
    events["size_class"] = pd.cut(events["area_km2"], SIZE_BINS, labels=SIZE_LABELS, right=False)
    events["duration_class"] = pd.cut(events["duration_days"], DUR_BINS, labels=DUR_LABELS)
    events["peat_event"] = np.where(events["pct_on_peat"] > 0, "touche la tourbe", "hors tourbe")
    events = gpd.GeoDataFrame(events, geometry=footprints.geometry.reindex(events.index), crs=DIST_CRS).to_crs(4326)

    # --- ajouter l'identifiant et la taille du feu à chaque détection ---
    fires_all_atts["fire_event_id"] = ev["fire_event_id"].reindex(fires_all_atts.index)
    for col, src in [("event_area_km2", "area_km2"), ("event_duration_days", "duration_days"),
                     ("event_n_detections", "n_detections")]:
        fires_all_atts[col] = fires_all_atts["fire_event_id"].map(events[src])

    n_peat = int((events["pct_on_peat"] > 0).sum())
    print(f"{len(ev):,} détections regroupées en {n_events:,} feux (distance ≤ {FIRE_LINK_DIST_M} m, écart ≤ {FIRE_MAX_GAP_DAYS} jour)")
    print(f"{n_peat:,} feux touchent la tourbe ; superficie médiane {events['area_km2'].median():.2f} km², plus grand feu {events['area_km2'].max():.1f} km²")

    # --- graphiques : nombre de feux par taille et par durée ---
    order = ["touche la tourbe", "hors tourbe"]
    cols = {"touche la tourbe": "#6b21a8", "hors tourbe": "#bdbdbd"}
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, cls_col, labels_, title, xlabel in [
        (axes[0], "size_class", SIZE_LABELS, "Nombre de feux par superficie", "Superficie des pixels détectés"),
        (axes[1], "duration_class", DUR_LABELS, "Nombre de feux par durée", "Durée (jours avec détections, du premier au dernier)"),
    ]:
        tab = (pd.crosstab(events[cls_col], events["peat_event"])
               .reindex(index=labels_, columns=order, fill_value=0))
        tab.plot(kind="bar", ax=ax, color=[cols[c] for c in order], width=0.8, rot=0)
        for cont in ax.containers:
            ax.bar_label(cont, labels=[f"{int(v):,}" if v else "" for v in cont.datavalues], padding=2, fontsize=8)
        ax.set_title(f"{title} : {COUNTRY}")
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Nombre de feux (échelle log)")
        ax.set_yscale("log")
        ax.legend(title="")
        ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()

    # --- tableau : les plus grands feux touchant la tourbe ---
    TOP_N = 20
    top = (events[events["pct_on_peat"] > 0]
           .sort_values("area_km2", ascending=False)
           .head(TOP_N)
           .drop(columns=["geometry", "size_class", "duration_class", "peat_event"]))
    top["start"] = top["start"].dt.strftime("%Y-%m-%d")
    top["end"] = top["end"].dt.strftime("%Y-%m-%d")
    display(HTML(f"<div style='font-weight:bold;font-size:13px;margin:6px 0 2px 0;'>Les {TOP_N} plus grands feux touchant la tourbe</div>"))
    display(top)

    # --- export des feux (emprises) en GeoJSON ---
    events_out = events.copy()
    for c in ["start", "end"]:
        events_out[c] = events_out[c].dt.strftime("%Y-%m-%d")
    for c in ["size_class", "duration_class"]:
        events_out[c] = events_out[c].astype(str)
    events_path = f"fire_events_{COUNTRY}_{start_date}_{end_date}.geojson"
    events_out.reset_index().to_file(events_path, driver="GeoJSON")
    print(f"💾 {len(events_out):,} feux exportés → {events_path}")


In [ ]:
# visualiser sur une carte
# --- S'assurer que les colonnes latitude/longitude existent ---
if not {"latitude", "longitude"}.issubset(fires_all_atts.columns):
    fires_all_atts["latitude"] = fires_all_atts.geometry.y
    fires_all_atts["longitude"] = fires_all_atts.geometry.x

# --- Paramètres d'allègement de la carte (chargement plus rapide) ---
HEAT_GRID_DEG = 0.01      # regroupement des feux sur une grille d'environ 1 km
SIMPLIFY_TOL = 0.005      # simplification des contours (environ 500 m)
MAX_OVERLAY_PX = 1500     # taille max. de l'image des classes de tourbière

# --- Points chauds : regrouper les feux par cellule de grille ---
# (une valeur par cellule, pondérée par le nombre de feux, au lieu d'un point par feu)
pts = fires_all_atts[["latitude", "longitude"]].dropna()
cells = (
    (pts / HEAT_GRID_DEG).round().mul(HEAT_GRID_DEG).round(4)
    .value_counts().reset_index(name="n")
)
weights = (cells["n"] / max(cells["n"].quantile(0.95), 1)).clip(upper=1)
heat_data = np.column_stack([cells["latitude"], cells["longitude"], weights]).tolist()
print(f"{len(pts):,} feux regroupés en {len(cells):,} cellules de {HEAT_GRID_DEG}°")

# --- Image des classes de tourbière réduite pour l'affichage ---
step = int(np.ceil(max(PEAT_RGBA.shape[:2]) / MAX_OVERLAY_PX))
peat_overlay = PEAT_RGBA[::step, ::step]

# --- Créer la carte ---
m = folium.Map(
    location=[fires_all_atts.latitude.mean(), fires_all_atts.longitude.mean()],
    zoom_start=6,
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery"
)

# --- Classes de tourbière (étape 0), ajoutées sous les points chauds de feux ---
folium.raster_layers.ImageOverlay(
    image=peat_overlay,
    bounds=PEAT_BOUNDS,
    mercator_project=True,
    opacity=0.6,
    name=f"Classes de tourbière ({COUNTRY})",
).add_to(m)

# --- Dégradé personnalisé bleu → rouge ---
custom_gradient = {
    0.0: "blue",
    0.25: "cyan",
    0.5: "lime",
    0.75: "yellow",
    1.0: "red"
}

# --- Ajouter les points chauds de feux ---
HeatMap(
    heat_data,
    name="Points chauds de feux",
    radius=6,
    blur=10,
    min_opacity=0.4,
    gradient=custom_gradient
).add_to(m)

# --- Ajouter les limites ADM1 (sans remplissage, contours blancs) ---
if "ADM1" in locals() and ADM1 is not None:
    adm_light = ADM1[["geometry"]].copy()
    adm_light["geometry"] = adm_light.geometry.simplify(SIMPLIFY_TOL, preserve_topology=True)
    folium.GeoJson(
        adm_light,
        name="Limites ADM1",
        style_function=lambda x: {
            "color": "white",
            "weight": 1,
            "fill": False
        }
    ).add_to(m)

# --- Ajouter l'AOI (optionnel) ---
if "AOI" in locals() and AOI is not None:
    aoi_light = AOI[["geometry"]].copy()
    aoi_light["geometry"] = aoi_light.geometry.simplify(SIMPLIFY_TOL, preserve_topology=True)
    folium.GeoJson(
        aoi_light,
        name="AOI",
        style_function=lambda x: {"color": "blue", "weight": 2, "fill": False}
    ).add_to(m)

# --- Ajouter la légende ---
legend_html = """
<div style="
position: fixed;
bottom: 10px; left: 10px; width: 200px; height: 90px;
background-color: white; border:2px solid grey; z-index:9999; font-size:13px;
padding: 10px;">
<b>🔥 Densité de feux</b><br>
<svg width="180" height="15">
  <defs>
    <linearGradient id="grad" x1="0%" y1="0%" x2="100%" y2="0%">
      <stop offset="0%" style="stop-color:blue;stop-opacity:1" />
      <stop offset="25%" style="stop-color:cyan;stop-opacity:1" />
      <stop offset="50%" style="stop-color:lime;stop-opacity:1" />
      <stop offset="75%" style="stop-color:yellow;stop-opacity:1" />
      <stop offset="100%" style="stop-color:red;stop-opacity:1" />
    </linearGradient>
  </defs>
  <rect width="180" height="15" fill="url(#grad)" />
</svg><br>
Faible&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Élevée
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --- Légende des classes de tourbière (en bas à droite) ---
peat_legend_items = "".join(
    f'<div><span style="display:inline-block;width:12px;height:12px;background:{PEAT_CLASS_COLORS[k]};'
    f'opacity:0.8;margin-right:6px;"></span>{v}</div>'
    for k, v in PEAT_CLASS_NAMES.items()
)
m.get_root().html.add_child(folium.Element(
    f'<div style="position:fixed;bottom:10px;right:10px;z-index:9999;background:white;'
    f'padding:8px 10px;border:2px solid grey;font-size:12px;">'
    f'<b>Classes de tourbière : {COUNTRY}</b>{peat_legend_items}</div>'
))

# --- Emprises des feux touchant la tourbe (section « Taille des feux ») ---
if "events" in globals() and len(events):
    pe = events[events["pct_on_peat"] > 0][["geometry", "start", "end", "duration_days", "area_km2", "n_detections"]].copy()
    pe["start"] = pe["start"].dt.strftime("%Y-%m-%d")
    pe["end"] = pe["end"].dt.strftime("%Y-%m-%d")
    pe["geometry"] = pe.geometry.simplify(0.001)
    folium.GeoJson(
        pe, name="Feux touchant la tourbe (emprises)", show=False,
        style_function=lambda x: {"color": "#ff00ff", "weight": 1, "fillOpacity": 0.35},
        tooltip=folium.GeoJsonTooltip(
            fields=["start", "end", "duration_days", "area_km2", "n_detections"],
            aliases=["Début", "Fin", "Durée (jours)", "Superficie (km²)", "Détections"],
        ),
    ).add_to(m)

# --- Contrôle des couches ---
folium.LayerControl().add_to(m)

show_map(m)

In [ ]:
# colonne « coords » (latitude,longitude) pour Looker Studio : type de champ Géo > Latitude, Longitude
_pts = fires_all_atts.geometry.to_crs(4326)
fires_all_atts["coords"] = [f"{y:.6f},{x:.6f}" for x, y in zip(_pts.x, _pts.y)]

# les feux sont déjà limités à la zone d'analyse (tourbe + ANALYSIS_MAX_DIST_KM)
fires_all_atts_filt = fires_all_atts.copy()
print(f"Feux exportés ({COUNTRY}) : {len(fires_all_atts_filt):,} (sur la tourbe ou à moins de {ANALYSIS_MAX_DIST_KM:g} km)")
print(fires_all_atts_filt[["latitude", "longitude", "coords"]].head(3))

#### 4. télécharger les résultats sur Google Drive

In [ ]:
# autoriser le téléchargement sur Google Drive (jeton du compte Google connecté à SEPAL)
cred_path = Path.home() / ".config/earthengine/credentials"
drive = None
if cred_path.exists():
    with open(cred_path, "r") as f:
        access_token = json.load(f).get("access_token")
    drive = build('drive', 'v3', credentials=Credentials(token=access_token))
    print("✅ Connexion à Google Drive prête")
else:
    print("⚠️ Aucun jeton Google trouvé (~/.config/earthengine/credentials).\n"
          "   Connectez votre compte Google dans SEPAL, redémarrez le noyau, puis relancez cette cellule.\n"
          "   En attendant, les résultats sont enregistrés localement et le téléchargement vers Google Drive est ignoré.")

In [ ]:
# exporter en CSV
CSV_PATH = f"/home/sepal-user/Peatlands_Congo/fires_{COUNTRY}_{start_date}_{end_date}.csv"
# Exporter d'abord en CSV dans SEPAL
fires_all_atts_filt.to_csv(CSV_PATH, index_label='ID')

In [ ]:
if drive is None:
    print("⏭️ Google Drive non connecté : étape ignorée (voir le message de la cellule d'autorisation).")
else:
    # S'assurer que le dossier Peatlands existe
    folder_name = "Congo_Peatlands"
    folder_results = drive.files().list(
        q=f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields="files(id, name)"
    ).execute()
    folders = folder_results.get("files", [])

    if not folders:
        print(f"⚠️ Dossier '{folder_name}' introuvable. Création en cours...")
        folder_metadata = {"name": folder_name, "mimeType": "application/vnd.google-apps.folder"}
        folder = drive.files().create(body=folder_metadata, fields="id").execute()
        folder_id = folder["id"]
        print(f"🆕 Dossier '{folder_name}' créé (ID : {folder_id})")
    else:
        folder_id = folders[0]["id"]
        print(f"📁 Dossier '{folder_name}' trouvé (ID : {folder_id})")

In [ ]:
# même nom que le fichier CSV (sans l'extension)
SHEET_NAME = f"fires_{COUNTRY}_{start_date}_{end_date}"
print("Google Sheet :", SHEET_NAME)

In [ ]:
fires_all_atts_filt.to_csv(CSV_PATH, index_label="ID")

print(f"📁 CSV exporté localement : {CSV_PATH}")

if drive is None:
    print("⏭️ Google Drive non connecté : étape ignorée (voir le message de la cellule d'autorisation).")
else:
    # -------------------------------------------------------
    # Téléchargement sur Google Drive
    # -------------------------------------------------------
    VIIRS_file_metadata = {
        "name": SHEET_NAME.replace(".csv", ""),
        "mimeType": "application/vnd.google-apps.spreadsheet",
        "parents": [folder_id]
    }

    media = MediaFileUpload(CSV_PATH, mimetype="text/csv", resumable=True)

    # -------------------------------------------------------
    # Vérifier si le fichier existe déjà
    # -------------------------------------------------------
    existing = drive.files().list(
        q=f"name='{SHEET_NAME.replace('.csv','')}' and '{folder_id}' in parents and trashed=false",
        fields="files(id,name)"
    ).execute()

    files = existing.get("files", [])

    # -------------------------------------------------------
    # Remplacer ou télécharger
    # -------------------------------------------------------
    if files:
        file_id = files[0]["id"]
        print(f"♻️ Remplacement de la feuille Google existante '{SHEET_NAME}'...")

        updated = drive.files().update(
            fileId=file_id,
            media_body=media
        ).execute()

        file_id = updated["id"]

    else:
        uploaded = drive.files().create(
            body=VIIRS_file_metadata,
            media_body=media,
            fields="id"
        ).execute()

        file_id = uploaded["id"]

    print(f"✅ '{SHEET_NAME}' téléchargé dans le dossier Google Drive '{folder_name}'")

    # -------------------------------------------------------
    # Afficher un grand lien cliquable
    # -------------------------------------------------------
    sheet_url = f"https://docs.google.com/spreadsheets/d/{file_id}/edit"

    display(HTML(f"""
    <div style="margin-top:20px;">
    <b>Ouvrir les données de feux dans Google</b><br>
    <a href="{sheet_url}" target="_blank"
    style="font-size:22px;font-weight:bold;color:#1a73e8;">
    {sheet_url}
    </a>
    </div>
    """))

#### 5. exporter en GeoJSON pour Ground les feux sur la tourbe ou à moins de 1 km

In [ ]:
# --- Filtrer les feux sur la tourbe (PEAT_CODES) ou à moins de GROUND_MAX_DIST_KM de la tourbe ---
GROUND_MAX_DIST_KM = 1.0      # distance maximale à la tourbe (km) ; 0 = feux sur la tourbe uniquement

on_peat = fires_all_atts["peat_class_code"].isin(PEAT_CODES)
near_peat = fires_all_atts["dist_peat_km"] <= GROUND_MAX_DIST_KM
# --- Période exportée : les GROUND_LAST_DAYS derniers jours jusqu'à end_date (incluse) ---
end_dt = pd.Timestamp(end_date)
ground_start = (max(end_dt - pd.Timedelta(days=GROUND_LAST_DAYS - 1), pd.Timestamp(start_date))
                if GROUND_LAST_DAYS else pd.Timestamp(start_date))
acq_day = pd.to_datetime(fires_all_atts["acq_date"], errors="coerce").dt.normalize()
in_window = (acq_day >= ground_start) & (acq_day <= end_dt)

fires_peat = fires_all_atts[(on_peat | near_peat) & in_window].copy()
GROUND_START = f"{ground_start:%Y-%m-%d}"

print(f"✅ {len(fires_peat):,} feux sélectionnés ({COUNTRY}, du {GROUND_START} au {end_date}) : "
      f"{int((on_peat & in_window).sum()):,} sur la tourbe, "
      f"{int((near_peat & ~on_peat & in_window).sum()):,} à moins de {GROUND_MAX_DIST_KM:g} km de la tourbe")

# --- Garder peat_class_name, supprimer le code numérique ---
fires_peat.drop(columns=["peat_class_code"], inplace=True, errors="ignore")

# --- Ajouter la propriété 'name' à partir de la date d'acquisition ---
fires_peat["name"] = fires_peat["acq_date"].astype(str).apply(
    lambda d: f"Feu_{d[:10]}" if pd.notna(d) and len(d) >= 10 else "Feu_inconnu"
)

# colonnes catégorielles -> texte pour le pilote GeoJSON
for c in fires_peat.columns:
    if isinstance(fires_peat[c].dtype, pd.CategoricalDtype):
        fires_peat[c] = fires_peat[c].astype(str)

if fires_peat.empty:
    print(f"⚠️ Aucun feu sur la tourbe ou à moins de {GROUND_MAX_DIST_KM:g} km pour {COUNTRY} entre {GROUND_START} et {end_date} : aucun fichier GeoJSON exporté.")
else:
    GEOJSON_PATH = f"fires_peatland_{COUNTRY}_{GROUND_START}_{end_date}.geojson"
    fires_peat.to_crs(4326).to_file(GEOJSON_PATH, driver="GeoJSON")
    print(f"💾 {len(fires_peat):,} feux exportés pour {COUNTRY} → {GEOJSON_PATH}")